# DSC 80 - Discussion 03

---


In [ ]:
# import libraries
import pandas as pd
import numpy as np
import os
from IPython.display import HTML


In [ ]:
def multi_table(table_list):
    ''' Acceps a list of IpyTable objects and returns a table which contains each IpyTable in a cell
    '''
    return HTML(
        '<table><tr style="background-color:white;">' + 
        ''.join(['<td>' + table._repr_html_() + '</td>' for table in table_list]) +
        '</tr></table>'
    )




# Review: Hypothesis Testing & Combining DataFrames

## Hypothesis Testing

We will cover examples of two key types of hypothesis testing
* Comparing two categorical distributions
* Comparing sub-group statistic to a population parameter

### Steps to follow to solve a hypothesis testing problem
1. Form the Null and Alternate hypothesis
2. Define the test statistic
3. Calculate observed/sample test statistic (form an intuition of how big/small/extreme it is)
4. Simulate one instance of test statistic under null hypothesis. Simulate the whole null distribution
5. Calculate p-value based on null distribution and observed test statistic
6. Plot the null distribution, observed statistic and validate the p-value
7. Provide conclusion as to whether you reject or fail-to-reject null hypothesis

### Are Snapchat users similar in age distribution compared to overall social media users?
Based on the (fake) survey collected from 1000 Snapchat users, we want to understand if the age distribution of these two user groups are significantly different?
- Note that the survey data below is already aggregated, which can be used directly.

In [ ]:
age_df = pd.DataFrame([['GenZ', 0.64, 0.48],
                    ['Millennials', 0.24, 0.36],
                    ['GenX', 0.08, 0.10],
                    ['Boomers', 0.04, 0.06]],
                   columns=['Age Group', 'Snapchat', 'Social Media']).set_index('Age Group')

age_df


#### Step-1: Form the Null and Alternate Hypotheses
Null hypothesis:

Alternate hypothesis: 

#### Step-2: Define the test-statistic

A common test statistic used while comparing two categorical distributions is TVD (Total Variation Distance)

**TVD:** Sum of absolute differences between corresponding values of two distribution, divided by 2.

### Why TVD? 

Why not just subtract the GenZ percentages?

<details>
<summary>🔍 Show solution</summary>
Because we have multiple categories. If we just look at GenZ, we ignore what's happening to Millennials or Boomers. 

TVD gives us a single number to describe the "distance" between two entire probability distributions.

</details>

#### Step-3: Calculate observed/sample test statistic

In [ ]:

age_df.plot(kind='barh', title='Age Group Distribution of Snapchat vs Social Media');


In [ ]:
observed_tvd = np.sum(np.abs(age_df['Snapchat'] - age_df['Social Media'])) / 2
observed_tvd

# Think - What would be the observed TVD if the two groups are exactly similar?
# Further - What does this imply about the TVD value as the difference increases/decreases?

<details>
<summary>🔍 Show solution</summary>

“What does this imply about the TVD value as the difference increases/decreases?”: **As the two distributions diverge, the TVD increases; as they become more similar, the TVD decreases towards 0.**

</details>

#### Step-4 Simulate the test-statistic under null hypothesis. And simulate the null distribution

In [ ]:
# Simulate one instance - Keep null hypothesis in mind
# There is no difference between Snapchat age groups wrt. overall age groups.
# So, a sample under null hypothesis should like an 'overall social media' distribution.
np.random.multinomial(1000, age_df['Social Media']) / 1000

In [ ]:
# Simulate N times - Larger the N, smoother will be the Null distribution
# Traditional way - Use a for loop and store the null test statistic
# Faster way - Utilize numpy functionalities
N = 5000
np.random.multinomial(1000, age_df['Social Media'], size=N) / 1000

In [ ]:
null_dists = np.random.multinomial(1000, age_df['Social Media'], size=N) / 1000

# Power of broadcasting: Can take difference between 2D array and 1D array 
# by broadcasting 1D array to 2D array
null_tvds = np.sum(np.abs(null_dists - age_df['Social Media'].to_numpy()), axis=1) / 2
null_tvds

#### vectorizing code

In [ ]:
# The "Slow" Way (Iterative Loop)
N = 5000
null_tvds_loop = []

for i in range(N):
    # 1. Simulate one instance under the Null
    sample = np.random.multinomial(1000, age_df['Social Media']) / 1000
    
    # 2. Calculate the TVD for this single instance
    # Formula: 1/2 * sum(|p_i - q_i|)
    tvd = np.sum(np.abs(sample - age_df['Social Media'].values)) / 2
    
    # 3. Append to list
    null_tvds_loop.append(tvd)

null_tvds_loop = np.array(null_tvds_loop)


In [ ]:
N = 5000

# 1. Simulate all 5000 trials in one (N, 4) matrix
null_sims = np.random.multinomial(1000, age_df['Social Media'], size=N) / 1000

null_sims.shape


In [ ]:
age_df['Social Media'].shape

In [ ]:
# 2. Vectorized TVD calculation using broadcasting
# Aligning a (5000, 4) matrix with a (4,) vector
null_tvds_vec = np.sum(np.abs(null_sims - age_df['Social Media'].values), axis=1) / 2

In [ ]:
# The "Delta" check
difference = null_tvds_loop - null_tvds_vec

print(f"Mean Difference: {np.mean(difference):.6f}")
print(f"Are they identical? {np.array_equal(null_tvds_loop, null_tvds_vec)}")

You'll notice the subtraction isn't zero. This isn't a bug in the vectorization, and it’s not a rounding error. 

It’s because simulation is stochastic.
Every time we call np.random, we are drawing a new path through a random process.
The loop and the vector are essentially two different 'universes' of 5,000 trials.

#### Step-5: Calculate p-value

In [ ]:
p_val = np.mean(null_tvds >= observed_tvd)
p_val

#### Step-6: Visualize and validate the p-value

In [ ]:
import matplotlib.pyplot as plt
pd.Series(null_tvds).plot(kind='hist', 
                     density=True,
                     ec='w',
                     title='Simulated Null TVDs & Observed TVD');
plt.axvline(x=observed_tvd, color='red', linewidth=2);

In [ ]:
# Recall Null Hypothesis - 

#  Think - What happens to your belief about the null hypothesis if we assume the distribution is taken 
#  from only 10 or 100 users?
#  Would you believe the null hypothesis more or believe it less??

<details>
<summary>🔍 Show solution</summary>


Would you believe the null hypothesis more or believe it less??: **You would (rationally) tend to believe (fail to reject) the null hypothesis more often with smaller sample sizes because of increased variability in the null distribution.**

“from only 10 or 100 users?”: **With smaller sample sizes (e.g. only 10 or 100 users), the null distribution is more variable, so an observed statistic is less extreme relative to it, leading to a higher p‑value and making us less likely to reject the null hypothesis.**

</details>

#### Step-7: Provide conclusion

Conclusion: 

**Question 1**

A study from a competitive programming test shows a lower average score for students who used 'Go' programming language compared to others. Test whether this lower avg. score of 'Go' programming language is due to chance alone.

- The function should take a DataFrame like `prog_df`, number of null test statistic simulations `N`, 
- It should return a list containing a) observed test statistic, b) and the p-value of the hypothesis test. Note that the function should work for any `prog_df` having same column names, and the same unique values in `language` column.

Hint: Don't forget to utilize `numpy` functionality to eliminate the `for` loop. 

In [ ]:
prog_df = pd.read_csv(os.path.join('data','prog_df.csv'))
grouped_df = prog_df.groupby('language')['score'].agg(['mean', 'count'])
grouped_df

In [ ]:
# What's the test statistic?
# What's the observed test statistic?
# How to simulate the null hypothesis?
# p-value?

<details>

<summary>Solution</summary>

What's the test statistic?: **The test statistic is the Total Variation Distance (TVD) between the two categorical distributions.**

“What's the observed test statistic?”: The observed TVD calculated from the sample

“How to simulate the null hypothesis?”: Simulate counts under the null by drawing from a multinomial distribution parameterized by the overall social media proportions, then compute the TVD for each draw.

“p-value?”: is the p‑value is the fraction of simulated TVDs greater than or equal to the observed TVD?

</details>

<details>
<summary>🔍 Show solution code</summary>

```python
def hyp_test_lower_avg(df, N):
    """
    Tests whether the lower avg. of 'Go' is due to chance alone.
    
    - The function should take a DataFrame like prog_df, number of null test statistic simulations N,
    - It should return a list containing a) observed test statistic, b) and the p-value of the hypothesis test

    :Example:
    >>> prog_df = pd.read_csv(os.path.join('data','prog_df.csv'))
    >>> q1_out = hyp_test_lower_avg(prog_df, 1000)
    >>> isinstance(q1_out, list)
    True
    >>> np.isclose(q1_out[0], 76.96903195339905, atol=0.01)
    True
    >>> 0.08 <= q1_out[1] <= 0.20
    True
    """
    # BEGIN SOLUTION
    grouped_df = df.groupby('language')['score'].agg(['mean', 'count'])
    n_go = grouped_df.loc['Go', 'count']
    observed_go_avg = grouped_df.loc['Go', 'mean']
    
    null_go_avgs = np.random.choice(df['score'], size=(N, n_go)).mean(axis=1)
    
    p_val = np.mean(null_go_avgs <= observed_go_avg)
    
    return [observed_go_avg, p_val]
    # END SOLUTION
```
</details>

In [ ]:
# don't change this cell -- it is needed for the tests to work

prog_df = pd.read_csv(os.path.join('data','prog_df.csv'))
q1_out = hyp_test_lower_avg(prog_df, 1000)

## Combining DataFrames 

* [`concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
* [`merge()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
* [`join()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)

### `concat()`

* Used to append one (or more) dataframes one below the other (or sideways, depending on whether the axis option is set to 0 or 1).
    * Useful if we have two or more data sets containing the same columns but different rows of data.
    * We can also concat the columns from one `Dataframe` to those of another `Dataframe`.

In [ ]:
# left dataframe
left = pd.DataFrame({
   'id':[1,2,3,4,5],
   'Name': ['Aaron', 'Marina', 'Justin', 'Janine', 'Ilya'],
   'subject_id':['sub1','sub2','sub4','sub6','sub5']})

# right dataframe
right = pd.DataFrame(
   {'id':[1,2,3,4,5],
   'Name': ['Enrique', 'Parker', 'Erik', 'Allston', 'Betty'],
   'subject_id':['sub2','sub4','sub3','sub6','sub5']})

multi_table([left, right])

In [ ]:
# add 'left' below 'right'
pd.concat([right, left])

In [ ]:
# if you want to keep track of the names dataframes after concat, use 'keys'
pd.concat([right, left], keys=['right', 'left'])

In [ ]:
# add 'left' to the side of 'right'
pd.concat([right, left], axis=1)

### `merge()`

* Used to combine two (or more) dataframes on the basis of **values of common columns** (indices can also be used, use `left_index=True` and/or `right_index=True`).
    * If we are joining columns on columns, the DataFrame indexes will be ignored. 
    * If we are joining indexes on indexes or indexes on a column or columns, the index will be passed on.

* **`on`**: column or index names to join on. 
    * These must be found in both DataFrames. 
    * If `on` is `None` and not merging on indexes, then this defaults to the intersection of the columns in both DataFrames.

In [ ]:
multi_table([left, right])

In [ ]:
# merge left and right tables on 'id' column
on_id = pd.merge(left,right,on='id')

# how many rows, how many columns?
multi_table([left, right, on_id])

In [ ]:
# merge left and right tables on 'id' and 'subject_id' columns
on_id_subject = pd.merge(left,right,on=['id', 'subject_id'])

# how many rows, how many columns, what are the indices?
multi_table([left, right, on_id_subject])

* **`how`**: specifies how to determine which keys are to be included in the resulting table. 
    * If a key (column name) combination does not appear in either the left or the right tables, the values in the joined table will be `np.NaN`.
    * Defaults to `inner`

In [ ]:
data_a = {
        'subject_id': ['1', '2', '3', '4', '5'],
        'first_name': ['Manny', 'Will', 'Hunter', 'Ian', 'Eric'], 
        'last_name': ['Machado', 'Myers', 'Renfroe', 'Kinsler', 'Hosmer']}
df_a = pd.DataFrame(data_a, columns = ['subject_id', 'first_name', 'last_name'])

data_b = {
        'subject_id': ['4', '5', '6', '7', '8'],
        'first_name': ['Cody', 'Justin', 'Corey', 'Clayton', 'Kenley'], 
        'last_name': ['Bellinger', 'Turner', 'Seager', 'Kershaw', 'Jansen']}
df_b = pd.DataFrame(data_b, columns = ['subject_id', 'first_name', 'last_name'])

multi_table([df_a, df_b])

| Merge Method  | Description                  |
| :-------      | :---------------------------:| 
| `left`        | Use keys from left object    | 
| `right`       | Use keys from right object   | 
| `outer`       | Use union of keys            |
| `inner`       | Use intersection of keys     | 

In [ ]:
# based on the output below, what 'how' argument was passed into pd.merge?
how_list = ['outer', 'inner', 'right', 'left']

merge_method = np.random.choice(how_list)

pd.merge(df_a, df_b, on='subject_id', how=merge_method)

In [ ]:
# let's check!
merge_method

### `join()`

* Used to merge two dataframes on the basis of the index; instead of using `merge()` with the option `left_index=True`, we can use `join()`.
    * Join operation honors the object on which it is called: `a.join(b)` $ \neq $ `b.join(a)`.

Facts about `merge()` vs `join()`:

* `merge()` is the underlying function used for all merge/join behavior
* `join()` is basically a specific behavior of `merge()` (left join using indices)
* For all practical purposes, `merge()` is usually used. While speaking, in general, merging and joining mean the same thing - combining DataFrames/tables based on common columns or indices.

<img src="imgs/join_types.jpg">

1. **Inner Join** – only keep rows where the merge “on” value exists in both the left and right dataframes.
2. **Left Outer** – keep every row in the left dataframe.
    * Where there are missing values of the “on” variable in the right dataframe, add `np.NaN` values in the result.
3. **Right Join** – keep every row in the right dataframe. 
    * Where there are missing values of the “on” variable in the left column, add `np.NaN` values in the result.
4. **Outer Join** – returns all the rows from the left dataframe, all the rows from the right dataframe, and matches up rows where possible, with `NaNs` elsewhere.

We'll start with a simple example:

In [ ]:
df1 = pd.DataFrame({'key': ['foo', 'bar'], 'val': [1, 2]}).set_index('key')
df2 = pd.DataFrame({'key': ['foo', 'bar'], 'val': [4, 5]}).set_index('key')

joined = df1.join(df2, lsuffix='_l', rsuffix='_r')

multi_table([df1, df2, joined])

Now let's try something a bit more complex:

In [ ]:
df1_data = {
    'Year' : [2014, 2014, 2014, 2014, 2014],
    'Week' : ['A', 'B', 'B', 'C', 'D'],
    'Color' : ['Red', 'Red', 'Black', 'Red', 'Green'],
    'Val' : [50, 60, 70, 10, 20]
}

df1 = pd.DataFrame(df1_data).set_index('Week')

df2_data = {
    'Year' : [2014, 2014, 2014, 2014, 2014],
    'Week' : ['A', 'B', 'C', 'C', 'D'],
    'Color' : ['Black', 'Black', 'Green', 'Red', 'Red'],
    'Score' : [30, 100, 50, 20, 40]
}

df2 = pd.DataFrame(df2_data).set_index('Week')

multi_table([df1, df2])

In [ ]:
# how many rows, how many columns?
df1.join(df2, lsuffix='_l', rsuffix = '_r')

In [ ]:
# will this be any different?
df2.join(df1, lsuffix='_l', rsuffix = '_r')

### Sample Midterm Question
How many rows will you get if you perform:
- a) `df1` **left join** `df2` on `'letter'` ?
- b) `df1` **inner join** `df2` on `'letter'` ?
- c) `df1` **right join** `df2` on `'letter'` ?

Answer the question without using python code.
Can you write how the final merged/joined table will look like?

In [ ]:
df1 = pd.DataFrame({
    'letter' : [1, 1, 2, 3, 4, 4],
    'alphabet' : ['A', 'B', 'C', 'D', 'E', 'F']
})

df2 = pd.DataFrame({
    'letter' : [1, 2, 4, 4, 4],
    'alphabet' : ['G', 'H', 'I', 'J', 'K']
})

multi_table([df1, df2])

**Question 2**

You are given two seperate dataframes: `mlb_2017` and `mlb_2018`. Both dataframes contain statistics for the 2017 and 2018 baseball seasons respectively. Your job is two combine these two dataframes into one using the following guidelines:

* The dataframe you return should be indexed by team name (`Tm`).
* The dataframe you return should include all columns from both `mlb_2017` and `mlb_2018`.
* Use the suffixes `_2017` and `_2018` to differentiate between statistics from both seasons.

Create a function `combined_seasons` that returns, as a tuple, the following:

* The combined dataframe described above.
* The team with most homeruns (`HR`) bewteen the 2017 and 2018 seasons combined.

In [ ]:
# read in the following .txt files
mlb_2017 = pd.read_csv(os.path.join('data','mlb_2017.txt'))
mlb_2018 = pd.read_csv(os.path.join('data','mlb_2018.txt'))

multi_table([mlb_2017.head(), mlb_2018.head()])

<details>
<summary>🔍 Show solution code</summary>

```python
def combined_seasons(df1, df2):
    """
    Create a function that return, as a tuple, a dataframe combining
    the 2017 and 2018 MLB seasons as well as the team that hit the most
    homeruns between the two seasons.

    :Example:
    >>> mlb_2017 = pd.read_csv(os.path.join('data','mlb_2017.txt'))
    >>> mlb_2018 = pd.read_csv(os.path.join('data','mlb_2018.txt'))
    >>> q2_out = combined_seasons(mlb_2017, mlb_2018)
    >>> q2_out[0].shape == (30, 56)
    True
    >>> q2_out[1] in q2_out[0].index
    True
    >>> all([(('_2017' in x) or ('_2018' in x)) for x in q2_out[0]])
    True
    >>> q2_out[1] == 'NYY'
    True
    """
    # BEGIN SOLUTION
    df_combined = pd.merge(df1, df2, on='Tm', suffixes=('_2017', '_2018')).set_index('Tm')
    
    hr_max = df_combined.filter(regex='HR').sum(axis=1).idxmax()
    
    return (df_combined, hr_max)
    # END SOLUTION
```
</details>

**Question 3**

Using the same two dataframes, `mlb_2017` and `mlb_2018`, create a function `seasonal_average` that combines them and takes the mean of each column for each team. 

* The dataframe you return should be indexed by team name (`Tm`).
* Each column should contain the average value between the *2017* and *2018* seasons for the given statistic for each team.
    * For example, the `HR` column should contain the average value for `HR` for each team between the *2017* and *2018* seasons.

<details>
<summary>🔍 Show solution code</summary>

```python
def seasonal_average(df1, df2):
    """
    Combines df1 and df2 and take the mean of each column 
    for each team.
    
    - The dataframe you return should be indexed by team name (Tm).
    - Each column should contain the average value between the 2017 
    and 2018 seasons for the given statistic for each team.

    :Example:
    >>> mlb_2017 = pd.read_csv(os.path.join('data','mlb_2017.txt'))
    >>> mlb_2018 = pd.read_csv(os.path.join('data','mlb_2018.txt'))
    >>> q3_out = seasonal_average(mlb_2017, mlb_2018)
    >>> q3_out.shape == (30, 28)
    True
    >>> q3_out.index.nunique() == 30
    True
    >>> q3_out.loc['MIN']['HR'] == 186
    True
    >>> q3_out.loc['OAK'].max() == 6190.5
    True
    """
    # BEGIN SOLUTION
    df_combined = pd.concat((df1, df2)).set_index('Tm')
    
    averages = df_combined.groupby(df_combined.index).mean()
    
    return averages
    # END SOLUTION
```
</details>

### Summary:

Left Join: Keep everything in the "Anchor" table (Left). If the Right table doesn't have a match, it’s a NaN.

Inner Join: Only the "Intersection." If either side is missing the key, that row is gone.

Many-to-Many: If you merge two tables that both have duplicate keys, you trigger a Cartesian product. You'll end up with more rows than you started with, which could break your downstream analysis. Check df.shape before and after a merge.

### Git Review

Git is for reproducibility, not just storage. We try to follow three best practices:

 -  Atomic Commits: One feature or one bug fix per commit. It makes git revert useful when a debugging is neded or your model breaks.

 -  Repo Hygiene: Use a .gitignore. We never track .ipynb_checkpoints, .DS_Store, or raw data files over 50MB. Large files belong in a cloud bucket, not your Git history.

 - The 'Stash' Workflow: If you're mid-experiment and need to pull a teammate's changes, don't do a messy commit. Use git stash, git pull, then git stash pop.

Essential commands:

#### git commit -m "Short, imperative summary"

#### git checkout -b <branch_name> 

#### git status, git pull, git push

## Congratulations! You're done!